# Preparação do ambiente e download das bases

Instala as dependências do projeto e disponibiliza em data/raw os arquivos de entrada usados nas etapas seguintes.


In [1]:
# Objetivo: Instala as dependencias listadas em requirements.txt, localizando a raiz do projeto a partir da pasta atual.

from pathlib import Path
import subprocess
import sys

# Localiza a raiz do projeto que contem o arquivo de dependências.
# Localiza a raiz do projeto que contem o arquivo de dependencias.
raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").exists())

# Instala no kernel atual os pacotes declarados pelo projeto.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-r",
    str(raiz / "requirements.txt"),
])

0

In [ ]:
# Objetivo: Baixa as bases necessarias para data/raw, ignora arquivos ja existentes e acompanha o progresso de cada download.

from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeout
from pathlib import Path
from time import perf_counter
from urllib.request import urlopen

# Define a pasta de dados de entrada e identifica os arquivos esperados no Drive.
root = Path.cwd().parent
destino = root / "data" / "raw"
destino.mkdir(parents=True, exist_ok=True)
arquivos = {
    '29_BA.csv': '1IXspjE9g_ZQiYBj07-qynXX5a2DK12h6',
    '2026_teste_tecnico_relinke_cidacs(in).csv': '1hEp6_VKqU0OEJpoFoKnJi5cfLH6tqokW',
}

# Prepara a pasta de destino e inicia o acompanhamento da execução.
# Prepara a pasta de destino e inicia o acompanhamento da execucao.
destino.mkdir(parents=True, exist_ok=True)
print('=== DOWNLOAD DAS BASES ===', flush=True)
print(f'Destino: {destino}', flush=True)
print(f'Arquivos previstos: {len(arquivos)}', flush=True)
inicio_execucao = perf_counter()

# Processa os arquivos um a um e pula aqueles que já foram baixados.
# Processa os arquivos um a um e pula aqueles que ja foram baixados.
for indice, (nome, arquivo_id) in enumerate(arquivos.items(), start=1):
    print(f'\n[{indice}/{len(arquivos)}] Verificando: {nome}', flush=True)
    arquivo = destino / nome
    if arquivo.exists() and arquivo.stat().st_size:
        print(f'  OK: arquivo ja existe ({arquivo.stat().st_size / 1e6:.2f} MB); download ignorado.', flush=True)
        continue

    # Baixa para um arquivo temporário, evitando deixar um arquivo final incompleto.
# Baixa para um arquivo temporario, evitando deixar um arquivo final incompleto.
temporario = arquivo.with_suffix(arquivo.suffix + '.part')
    print('  Arquivo ausente; iniciando conexao com o Google Drive...', flush=True)
    inicio_arquivo = perf_counter()
    url = f'https://drive.usercontent.google.com/download?id={arquivo_id}&export=download&confirm=t'
    # Aguarda a conexão em uma thread e informa o tempo de espera.
# Aguarda a conexao em uma thread e informa o tempo de espera.
with ThreadPoolExecutor(max_workers=1) as executor:
        resposta = executor.submit(urlopen, url, timeout=120)
        segundos_espera = 0
        while True:
            try:
                origem = resposta.result(timeout=1)
                break
            except FutureTimeout:
                segundos_espera += 1
                print(f'\r  Aguardando resposta do Drive... {segundos_espera}s', end='', flush=True)
    print('\r  Conexao estabelecida.                          ', flush=True)
    # Lê a resposta em blocos e grava o conteúdo enquanto atualiza o progresso.
# Le a resposta em blocos e grava o conteudo enquanto atualiza o progresso.
with origem, temporario.open('wb') as destino_temporario:
        total = int(origem.headers.get('Content-Length', 0))
        if total:
            print(f'  Conectado. Tamanho informado: {total / 1e6:.2f} MB.', flush=True)
        else:
            print('  Conectado. O servidor nao informou o tamanho; acompanhando bytes baixados.', flush=True)
        baixado = 0
        ultima_atualizacao = inicio_arquivo
        while bloco := origem.read(1024**2):
            destino_temporario.write(bloco)
            baixado += len(bloco)
            agora = perf_counter()
            if agora - ultima_atualizacao >= 0.5:
                decorrido = agora - inicio_arquivo
                velocidade = baixado / decorrido / 1e6
                if total:
                    progresso = min(baixado / total, 1)
                    barra = '#' * int(progresso * 30) + '-' * (30 - int(progresso * 30))
                    restante = (total - baixado) / (baixado / decorrido) if baixado else 0
                    print(f'\r  [{barra}] {progresso:6.1%} | {baixado / 1e6:,.1f}/{total / 1e6:,.1f} MB | {velocidade:,.1f} MB/s | ETA {restante:,.0f}s', end='', flush=True)
                else:
                    print(f'\r  {baixado / 1e6:,.1f} MB baixados | {velocidade:,.1f} MB/s | {decorrido:,.0f}s', end='', flush=True)
                ultima_atualizacao = agora
    # Promove o arquivo temporário a arquivo final e registra a velocidade.
# Promove o arquivo temporario a arquivo final e registra a velocidade.
temporario.replace(arquivo)
    duracao = perf_counter() - inicio_arquivo
    velocidade = arquivo.stat().st_size / duracao / 1e6
    print(f'\r  Concluido: {nome} ({arquivo.stat().st_size / 1e6:.2f} MB em {duracao:.1f}s; media {velocidade:.1f} MB/s).', flush=True)

print(f'\n=== EXECUCAO FINALIZADA em {perf_counter() - inicio_execucao:.1f}s ===', flush=True)

=== DOWNLOAD DAS BASES ===
Destino: c:\Users\Amanda\Pictures\Laceda\Case-Laceda\data\raw
Arquivos previstos: 2

[1/2] Verificando: 29_BA.csv
  Arquivo ausente; iniciando conexao com o Google Drive...
  Conexao estabelecida.                          
  Conectado. Tamanho informado: 1461.66 MB.
  Concluido: 29_BA.csv (1461.66 MB em 118.3s; media 12.4 MB/s).| 12.4 MB/s | ETA 0ss

[2/2] Verificando: 2026_teste_tecnico_relinke_cidacs(in).csv
  Arquivo ausente; iniciando conexao com o Google Drive...
  Conexao estabelecida.                          
  Conectado. Tamanho informado: 0.01 MB.
  Concluido: 2026_teste_tecnico_relinke_cidacs(in).csv (0.01 MB em 1.8s; media 0.0 MB/s).

=== EXECUCAO FINALIZADA em 120.2s ===
